%md
# Allelic Effect Dashboard

This notebook provides an interactive dashboard to explore allelic effects across genomic regions. It loads a summary dataset containing clustered loci information (e.g., trait, chromosome, region boundaries, number of tag loci, and fold-change effects) along with pre-generated visualization images.

Users can interactively:

Select a trait
Filter by chromosome
Browse genomic clusters

For each selected cluster, the notebook displays key statistics (such as fold change and number of tag loci) together with the corresponding precomputed plot.

The goal is to enable quick, intuitive inspection of allelic effect patterns without recomputing analyses, using preprocessed results and visual summaries.

In [0]:
# ============================================================
# IMAGE-BASED ALLELIC EFFECT DASHBOARD (FINAL, STABLE)
# ============================================================

import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, Image

# ------------------------------------------------------------
# 0) PATHS
# ------------------------------------------------------------

PNG_DIR = "/Volumes/bmqg/default_bronze/fatemeh/final_project/allelic_effect_plots_newharvested/"
CSV_PATH = "/Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs_newharvested/allelic_effect_csv_neharvested/allelic_effect_summary_newharvested.csv"


assert os.path.exists(CSV_PATH), "Summary CSV not found"
assert os.path.exists(PNG_DIR),  "PNG directory not found"

# ------------------------------------------------------------
# 1) LOAD SUMMARY CSV
# ------------------------------------------------------------

df = pd.read_csv(CSV_PATH)

# Normalize column names
df.columns = df.columns.str.strip().str.lower()

#  FIX: map correct fold-change column
assert "fold_change_4_vs_0" in df.columns, "Expected column 'fold_change_4_vs_0' not found"
df["fold_change"] = df["fold_change_4_vs_0"]

# Required columns check
required_cols = [
    "trait", "chrom", "start_min", "start_max",
    "n_taglos", "fold_change", "png"
]

missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

# Type casting
df["trait"] = df["trait"].astype(str)
df["chrom"] = df["chrom"].astype(str)
df["png"]   = df["png"].astype(str)

print("Rows:", len(df))
display(df.head())

# ------------------------------------------------------------
# 2) GLOBAL STATE
# ------------------------------------------------------------

_cluster_view = pd.DataFrame()

# ------------------------------------------------------------
# 3) WIDGETS
# ------------------------------------------------------------

trait_dd = widgets.Dropdown(
    options=sorted(df["trait"].unique()),
    description="Trait:",
    layout=widgets.Layout(width="420px")
)

chrom_dd = widgets.Dropdown(
    description="Chrom:",
    layout=widgets.Layout(width="260px")
)

cluster_dd = widgets.Dropdown(
    description="Cluster:",
    layout=widgets.Layout(width="900px")
)

out = widgets.Output()

# ------------------------------------------------------------
# 4) UPDATE FUNCTIONS
# ------------------------------------------------------------

def update_chroms(*_):
    sub = df[df["trait"] == trait_dd.value]
    chrom_dd.options = sorted(sub["chrom"].unique())
    if chrom_dd.options:
        chrom_dd.value = chrom_dd.options[0]


def update_clusters(*_):
    global _cluster_view

    _cluster_view = df[
        (df["trait"] == trait_dd.value) &
        (df["chrom"] == chrom_dd.value)
    ].reset_index(drop=True)

    cluster_dd.options = [
        (
            f"{r['chrom']}:{r['start_min']}-{r['start_max']} | "
            f"taglos={r['n_taglos']} | FC={r['fold_change']:.2f}",
            i
        )
        for i, r in _cluster_view.iterrows()
    ]

    # Reset selection
    cluster_dd.value = None
    if cluster_dd.options:
        cluster_dd.value = cluster_dd.options[0][1]


def show_plot(idx):
    if idx is None or len(_cluster_view) == 0:
        return

    with out:
        clear_output(wait=True)

        row = _cluster_view.iloc[idx]

        print(f"Trait      : {row['trait']}")
        print(f"Chrom      : {row['chrom']}")
        print(f"Region     : {row['start_min']} – {row['start_max']}")
        print(f"#Taglos    : {row['n_taglos']}")
        print(f"FoldChange : {row['fold_change']:.3f}")
        print(f"PNG        : {row['png']}")

        if os.path.exists(row["png"]):
            display(Image(filename=row["png"]))
        else:
            print(" PNG not found")

# ------------------------------------------------------------
# 5) WIDGET WIRING
# ------------------------------------------------------------

trait_dd.observe(update_chroms, names="value")
chrom_dd.observe(update_clusters, names="value")
cluster_dd.observe(lambda c: show_plot(c["new"]), names="value")

# ------------------------------------------------------------
# 6) INIT SEQUENCE
# ------------------------------------------------------------

update_chroms()
update_clusters()

if len(_cluster_view) > 0:
    show_plot(0)

# ------------------------------------------------------------
# 7) DISPLAY DASHBOARD
# ------------------------------------------------------------

display(
    widgets.VBox([
        trait_dd,
        chrom_dd,
        cluster_dd,
        out
    ])
)

Rows: 552


,trait,chrom,cluster_id,start_min,start_max,n_snps,n_taglos,taglo_ids,fold_change_4_vs_0,png,status,n_groups,dosages_used,spearman_rho,spearman_p,linear_slope,linear_r2,linear_p,delta_extreme,dosage_shape,fold_change
0,(E)-2-Decenal,ST4.03ch07,1,44350000,44650000,12,11,"336975,336993,336982,337231,337226,337228,3369...",NaN,/Volumes/bmqg/default_bronze/fatemeh/final_pro...,ok,3,"[2, 3, 4]",-0.5,0.666667,-872966.125,0.612986,0.427444,-1745932.25,non_monotonic,NaN
1,"1-Hexanol, 2-ethyl-",ST4.03ch07,1,44350000,44650000,12,11,"336975,336993,337227,336982,337231,337228,3369...",NaN,/Volumes/bmqg/default_bronze/fatemeh/final_pro...,ok,3,"[2, 3, 4]",-0.5,0.666667,-872966.125,0.612986,0.427444,-1745932.25,non_monotonic,NaN
2,1-Octen-3-ol,ST4.03ch00,1,11250000,11250000,1,4,"6476,6478,6475,6477",NaN,/Volumes/bmqg/default_bronze/fatemeh/final_pro...,ok,2,"[0, 1]",1.0,NaN,28880.000,1.000000,0.000000,28880.00,monotonic,NaN
3,1-Octen-3-ol,ST4.03ch00,2,21200000,21250000,2,2,"11481,11522",NaN,/Volumes/bmqg/default_bronze/fatemeh/final_pro...,ok,2,"[0, 2]",1.0,NaN,22674.250,1.000000,0.000000,45348.50,monotonic,NaN
4,1-Octen-3-ol,ST4.03ch00,3,33700000,33700000,5,4,"16446,16443,16444,16441",NaN,/Volumes/bmqg/default_bronze/fatemeh/final_pro...,ok,4,"[0, 1, 2, 3]",1.0,0.000000,25202.950,0.790874,0.110689,77973.50,monotonic,NaN
